# 🎓 MPCIM Thesis - Dual-Dimensional Predictive Analytics

**Title**: Dual-Dimensional Predictive Analytics untuk Career Progression
**Author**: Deni Sulaeman
**Date**: November 2025

## 📋 Project Overview

Predicting employee promotion using:
1. Performance Assessment
2. Behavioral Assessment
3. Psychological Assessment


## 🔧 1. Setup & Installation

In [ ]:
# Install packages (uncomment if needed)
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn xgboost shap imbalanced-learn openpyxl

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries imported successfully!")

In [ ]:
# Option 1: Use existing processed data
data_file = "../data/processed/full_dataset_processed.csv"

# Option 2: Specify your own data file path
# data_file = "../data/processed/your_data_file.csv"

# Option 3: For Google Colab users (uncomment below)
# from google.colab import files
# uploaded = files.upload()
# data_file = list(uploaded.keys())[0]

print(f"📂 Loading data from: {data_file}")

In [ ]:
# Separate features and target
y = df["has_promotion"]
X = df.drop(columns=["has_promotion", "employee_id_hash", "employee_id", "employee_name"], errors="ignore")

# Select only numeric features
X = X.select_dtypes(include=[np.number])

# Handle missing values
if X.isnull().sum().sum() > 0:
    print(f"⚠️ Missing values found: {X.isnull().sum().sum()}")
    X = X.fillna(X.median())
    print("✅ Missing values filled with median")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Apply SMOTE for balancing
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Training set: {X_train_scaled.shape}")
print(f"✅ Test set: {X_test_scaled.shape}")
print(f"✅ Features: {X.shape[1]}")
print(f"\nOriginal training distribution:")
print(y_train.value_counts())
print(f"\nBalanced training distribution:")
print(pd.Series(y_train_bal).value_counts())

## 🎯 11. Final Summary & Recommendations

### ✅ Analysis Complete!

This notebook has performed:
1. **Data Loading & Exploration** - Understanding the dataset
2. **Data Preprocessing** - Cleaning, scaling, and balancing
3. **Multiple Model Training** - 6 different ML algorithms
4. **Performance Comparison** - Comprehensive metrics evaluation
5. **Visualization** - Confusion matrices, ROC curves, feature importance
6. **Best Model Selection** - Based on F1-Score and other metrics

### 📊 Key Findings:
- Best performing model identified
- Important features for promotion prediction
- Model comparison across multiple metrics

### 🚀 Next Steps:
1. Fine-tune the best model with hyperparameter optimization
2. Deploy the model for production use
3. Monitor model performance over time
4. Collect feedback and iterate

---
**Author**: Deni Sulaeman  
**Project**: MPCIM Thesis - Dual-Dimensional Predictive Analytics  
**Date**: December 2025

In [ ]:
# Print classification reports for all models
print("📝 DETAILED CLASSIFICATION REPORTS")
print("=" * 70)

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    print(f"\n🤖 {name}")
    print("-" * 70)
    print(classification_report(y_test, y_pred, target_names=["Not Promoted", "Promoted"]))

print("=" * 70)
print("✅ All classification reports generated")

## 📝 10. Classification Reports

In [ ]:
# Feature importance for tree-based models
tree_models = ["Random Forest", "Gradient Boosting", "XGBoost"]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, name in enumerate(tree_models):
    model = trained_models[name]
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:10]
    
    axes[idx].barh(range(10), importances[indices], color="skyblue", edgecolor="navy")
    axes[idx].set_yticks(range(10))
    axes[idx].set_yticklabels([X.columns[i] for i in indices], fontsize=9)
    axes[idx].set_xlabel("Importance", fontsize=10)
    axes[idx].set_title(f"{name}\nTop 10 Features", fontweight="bold", fontsize=11)
    axes[idx].invert_yaxis()
    axes[idx].grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

print("✅ Feature importance analysis completed")

## 🔍 9. Feature Importance Analysis

In [ ]:
# Plot ROC curves for all models
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 8))

for name, model in trained_models.items():
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=2, label="Random Classifier")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves - All Models", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ ROC curves generated for all models")

## 📈 8. ROC Curves Comparison

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=["Not Promoted", "Promoted"],
                yticklabels=["Not Promoted", "Promoted"],
                cbar_kws={"label": "Count"})
    axes[idx].set_title(f"{name}\nConfusion Matrix", fontweight="bold", fontsize=11)
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xlabel("Predicted")

plt.tight_layout()
plt.show()

print("✅ Confusion matrices generated for all models")

## 🎯 7. Detailed Evaluation - Confusion Matrices

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results).T

print("📊 Model Performance Comparison:")
print("=" * 70)
display(results_df.style.highlight_max(axis=0, color="lightgreen"))

# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
results_df.plot(kind="bar", ax=axes[0], rot=45)
axes[0].set_title("Model Performance Comparison", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Score")
axes[0].set_xlabel("Model")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)

# Heatmap
sns.heatmap(results_df.T, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[1], cbar_kws={"label": "Score"})
axes[1].set_title("Model Metrics Heatmap", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Model")
axes[1].set_ylabel("Metric")

plt.tight_layout()
plt.show()

# Find best model
best_model_name = results_df["F1-Score"].idxmax()
print(f"\n🏆 Best Model (by F1-Score): {best_model_name}")
print(f"Performance:")
for metric, value in results[best_model_name].items():
    print(f"  • {metric}: {value:.4f}")

## 📊 6. Model Comparison & Visualization

In [ ]:
# Define multiple models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss", use_label_encoder=False),
    "SVM": SVC(probability=True, random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500)
}

results = {}
trained_models = {}

print("🚀 Training Multiple Models...")
print("=" * 70)

for name, model in models.items():
    print(f"\n🔄 Training: {name}")
    
    # Train model
    model.fit(X_train_scaled, y_train_bal)
    trained_models[name] = model
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_pred_proba)
    }
    
    print(f"  ✅ Accuracy: {results[name]['Accuracy']:.4f}")
    print(f"  ✅ F1-Score: {results[name]['F1-Score']:.4f}")
    print(f"  ✅ ROC-AUC: {results[name]['ROC-AUC']:.4f}")

print("\n" + "=" * 70)
print("✅ All models trained successfully!")

## 🤖 5. Model Training - Multiple Algorithms

## 🔧 4. Data Preprocessing

In [ ]:
# Load and explore data
df = pd.read_csv(data_file)
print(f"📊 Dataset Shape: {df.shape}")
print(f"📊 Columns: {df.columns.tolist()}")
print("\n" + "="*60)
print("First 5 rows:")
display(df.head())
print("\n" + "="*60)
print("Dataset Info:")
print(df.info())
print("\n" + "="*60)
print("Statistical Summary:")
display(df.describe())

# Check for target variable
if "has_promotion" in df.columns:
    print("\n" + "="*60)
    print("Target Distribution (has_promotion):")
    print(df["has_promotion"].value_counts())
    print("\nTarget Proportion:")
    print(df["has_promotion"].value_counts(normalize=True))

## 📊 3. Exploratory Data Analysis